<a href="https://colab.research.google.com/github/Surya0070305/Shoping_assistant_using_ReAct_loop/blob/main/ReAct_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
api_key=userdata.get('OPENAI_KEY')

In [3]:
from openai import OpenAI
client=OpenAI(api_key=api_key)

In [25]:
import json

In [74]:
def get_balance(balance):

  print('Checking balance!!!!')
  if not balance:
    print('You have no balance')
    return 0 # Return 0 or an appropriate default if no balance
  else:
    int_balance = int(balance) # Convert balance to integer before returning
    print(f'This is your current balance--->{int_balance}')
    return int_balance # Return the integer balance


In [20]:
def calculate_deficit(balance, price):
  print('Currently calculating the deficent amount that is remaining to purchase')
  deficient_amount=int(price)-int(balance)
  return print(f'This is the amount---> {deficient_amount} that you need to purchase this item')

In [34]:
def calculate_remaining(balance, price):
  # Note: This function expects 'balance' and 'price' to be numerical values (or convertible to int).
  # The issue where this function is not called when expected often stems from incorrect type handling (e.g., string comparison) in the calling function (like 'compare').
  print('Estimating the remaining amount that will be left after the purchase')
  remaining_amount=int(balance)-int(price)
  return print(f'This is the remaining amount---> {remaining_amount}. This amount will be remaining after the purchase')

In [52]:
def compare(balance, price):
  print('Comparing the cost of the product')
  balance_int = int(balance.strip())
  price_int = int(price.strip())
  if balance_int > price_int:
    calculate_remaining(balance_int, price_int)
  elif balance_int < price_int:
    calculate_deficit(balance_int, price_int)
  else:
    return print('You have no balance')

In [73]:
def suggest_more(balance):
  current_balance = get_balance(balance) # Now get_balance will return an integer
  if current_balance > 0:
    # Return a string message for the tool output
    return f'As you still have {current_balance} remaining, do you require assistance in suggesting any apparel or anything else?'
  else:
    return 'You have no balance left for further suggestions.'


In [58]:
Available_function={
    'get_balance':get_balance,
    'compare':compare,
    'calculate_remaining':calculate_remaining,
    'calculate_deficit':calculate_deficit,
    'suggest_more':suggest_more
}

In [57]:
tools_schema=[
    {
        "type": "function",
        "function": {
            "name": "get_balance",
            "description": "This function will check the balance available currently",
            "parameters": {
                "type": "object",
                "properties": {
                    "balance": {"type": "string", "description": "This field will give me the available amount in my current or saving or balance amount"}
                },
                "required": ["balance"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "compare",
            "description": "This function will check whether the purchase can be possible or not",
            "parameters": {
                "type": "object",
                "properties": {
                    "balance": {"type": "string", "description": "This field will give me the available amount in my current or saving or balance amount"},
                    "price": {"type": "string", "description": "This field will give me the price of the product"}
                },
                "required": ["balance","price"]
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "suggest_more",
            "description": "This function will help the shopper to have a proper suggestion done as the respective person is still having balance to contiue purchase",
            "parameters": {
                "type": "object",
                "properties": {
                    "balance": {"type": "string", "description": "This field will give me the available amount in my current or saving or balance amount"}
                },
                "required": ["balance"]
            }
        }
    }
]

In [76]:
def agent_run_loop():
  print('Welcome to the Shopping Assistant!')

  messages=[
      {'role':'system','content':'You are a helpful Shopping assistant, who can determine if an item can be purchased. Ensure numerical values are correctly handled as integers within functions for proper comparison. Also if after the purchase if the person is having any balance left or is he interested in continuing the shopping please suggest him the options for shopping'},
  ]
  quit_message=['quit', 'stop', 'exit', 'bye', 'no']

  while True:
    user_query = input("You: ") # Get user input for the current turn

    if user_query.lower() in quit_message:
        print("Goodbye!")
        break

    messages.append({'role':'user','content':user_query}) # Add user's message to the conversation history

    print('AI is Thinking----->')
    response = client.chat.completions.create(
        model="gpt-4o", #LLM
        messages=messages, #Memory + Prompt
        tools=tools_schema, #Tools
        tool_choice="auto"
    )

    response_msg = response.choices[0].message
    messages.append(response_msg) # Append the LLM's response (tool call or text)

    if response_msg.tool_calls: # If the LLM called a tool
      for tool_call in response_msg.tool_calls:
          func_name = tool_call.function.name
          func_args = json.loads(tool_call.function.arguments)

          function_to_call = Available_function.get(func_name)

          if function_to_call:
              tool_output = function_to_call(**func_args)

              messages.append({
                   "role": "tool",
                   "tool_call_id": tool_call.id,
                   "name": func_name,
                   "content": str(tool_output)
               })
          # The loop will reiterate after tool execution, allowing the LLM to process the tool outputs and generate a final text response.
    else: # If the LLM did NOT call a tool, it means it's a final text response
      print(f"\n[FINAL RESPONSE]: {response_msg.content}")
      # Removed 'break' to allow for continuous conversation until user explicitly quits.


In [75]:
agent_run_loop()

Welcome to the Shopping Assistant!
You: I currently have a balance of 10000 RS I want to purchase as shoe worth of 5000 rs, can i go ahead and purchase it
AI is Thinking----->
Comparing the cost of the product
Estimating the remaining amount that will be left after the purchase
This is the remaining amount---> 5000. This amount will be remaining after the purchase
You: Yes i will purchase it
AI is Thinking----->

[FINAL RESPONSE]: Great! You've purchased the shoes for 5000 RS. After this purchase, you have a remaining balance of 5000 RS.

Would you like to continue shopping and explore some more options?
You: yes
AI is Thinking----->
Checking balance!!!!
This is your current balance--->5000
As your are still having balance to continue purchase do you require assitance in suggestion of any apparel or anything
You: yes on tshirt, are there are any options
AI is Thinking----->

[FINAL RESPONSE]: Since you have a balance of 5000 RS, you can explore various t-shirt options within this budge